# 03 · Full CI via Slater-Condon Rules

FCI 是在给定基组下对完整 Hamiltonian 对角化得到的精确解。
本 notebook 从 Slater-Condon 规则出发用 Python 实现 FCI：

1. 手动做 AO->MO 积分变换 (einsum, 不考虑对称性打包, (nmo,nmo,nmo,nmo) 完整 4 维)
2. 用**位运算**表示 Slater 行列式
3. 用 **Slater-Condon 规则**计算矩阵元 <I|H|J>
4. 构建密集矩阵, `scipy.linalg.eigh` 对角化
5. 与 PySCF FCI 对比


---

## 1 · 积分准备: 手动 AO->MO 变换


In [16]:
import numpy as np
from scipy.linalg import eigh
from pyscf import gto, scf, fci as fci_mod

# -------- H2O / STO-3G --------
mol = gto.M(atom='O 0 0 0; H 0 -0.757 0.587; H 0 0.757 0.587', basis='sto-3g')
mf = scf.RHF(mol)
mf.kernel(verbose=0)

C = mf.mo_coeff                      # MO coefficients (nao, nmo)
norb = C.shape[1]                    # number of MOs
nelec = mol.nelectron                # 10 electrons
nalpha = nbeta = nelec // 2          # 5 alpha, 5 beta

# -------- AO integrals --------
hcore_ao = mf.get_hcore()            # core Hamiltonian (nao, nao)
eri_ao = mol.intor('int2e_sph')      # 2e integrals (nao,nao,nao,nao)
# pyscf convention: eri_ao[i,j,k,l] = (ij|kl) in chemist notation
E_nn = mf.energy_nuc()               # nuclear repulsion

print(f"norb={norb}, nelec={nelec}, nalpha={nalpha}, nbeta={nbeta}")
print(f"E_nn = {E_nn:.10f} Hartree")


converged SCF energy = -74.9630631297276
norb=7, nelec=10, nalpha=5, nbeta=5
E_nn = 9.1882584177 Hartree


---

## 2 · AO->MO 变换 (einsum, 不做对称性打包)

$$
h_{pq} = \sum_{\mu\nu} C_{\mu p} h_{\mu\nu}^{\text{AO}} C_{\nu q}
$$

$$
(pq|rs) = \sum_{\mu\nu\kappa\lambda} C_{\mu p} C_{\nu q} C_{\kappa r} C_{\lambda s} \, (\mu\nu|\kappa\lambda)^{\text{AO}}
$$

保留完整的 (norb, norb, norb, norb) 4 维数组。


In [17]:
# -------- h1e: MO 变换 (norb, norb) --------
h1e = C.T @ hcore_ao @ C

# -------- h2e: 4-index full transform (norb, norb, norb, norb) --------
h2e = np.einsum('up,vq,kr,ls,uvkl->pqrs', C, C, C, C, eri_ao, optimize=True)

print(f"h1e shape: {h1e.shape}")
print(f"h2e shape: {h2e.shape}")

# ecore = E_nn for full FCI
ecore = E_nn


h1e shape: (7, 7)
h2e shape: (7, 7, 7, 7)


---

## 3 · 位运算表示行列式

一个 Slater 行列式 = 两个整数: `alpha_bits` 和 `beta_bits`, 第 i 位=1 表示占据第 i 个轨道。

Python int 是任意精度, 轨道数无上限。


In [18]:
# -------- 位运算工具 --------
# Python int 任意精度, 对标 C++ <bit> 头文件

def popcount(x):
    # std::popcount
    return x.bit_count()

def ctz(x):
    # std::countr_zero -- 最低置位的索引. x 必须 > 0.
    return (x & -x).bit_length() - 1

def for_each_bit(bits, fn):
    # 遍历 bits 中每个置位, 调用 fn(index). 只遍历 k 个置位.
    while bits:
        p = ctz(bits)
        bits &= bits - 1
        fn(p)

def bits_to_list(bits):
    # 返回 bits 中所有置位的索引列表 (从小到大).
    result = []
    for_each_bit(bits, lambda p: result.append(p))
    return result

def count_between(bits, p, q):
    # bits 中严格位于 (p, q) 之间的 1 的个数 (不含 p, q).
    if p > q:
        p, q = q, p
    if q <= p + 1:
        return 0
    mask = ((1 << (q - p - 1)) - 1) << (p + 1)
    return (bits & mask).bit_count()

def sign(n):
    # (-1)^n
    return 1 if n % 2 == 0 else -1

# 演示
a = 0b0011010
print(f"alpha = {a:07b}")
print(f"  popcount = {popcount(a)}")
print(f"  bits_to_list = {bits_to_list(a)}")
print(f"  count_between(0, 3) = {count_between(a, 0, 3)}")
print(f"  count_between(1, 4) = {count_between(a, 1, 4)}")


alpha = 0011010
  popcount = 3
  bits_to_list = [1, 3, 4]
  count_between(0, 3) = 1
  count_between(1, 4) = 1


---

## 4 · Slater-Condon 规则

PySCF 积分约定: `eri[i,j,k,l] = (ij|kl)` (chemist notation)。

反对称化双电子积分: <pq||rs> = (pr|qs) - (ps|qr) = `h2e[p,q,r,s] - h2e[p,s,r,q]`

### Rule 1: 对角元

<I|H|I> = sum_{p in I} h_{pp} + 1/2 sum_{p,q in I} <pq||pq>

同自旋有交换项, 异自旋只有 Coulomb。

### Rule 2: 单激发 (ndiff=2, 差 1 个自旋轨道 p->q)

<I|H|J> = Gamma_p^q [ h_{pq} + sum_{r in I, same spin} <pr||qr> + sum_{r in I, opp spin} <pr|qr> ]

Gamma_p^q = (-1)^{count_between(p,q)}

### Rule 3: 双激发 (ndiff=4, 差 2 个自旋轨道 p->q, r->s)

同自旋: <I|H|J> = Gamma_p^q * Gamma_r^s * <pr||qs>
异自旋: <I|H|J> = Gamma_p^q * Gamma_r^s * <pr|qs>

### Rule 4: 三激发及以上 = 0


---

## 5 · Slater-Condon 实现


In [19]:
# ============================================================
# 激发检测: 用 bits_to_list (内部 ctz + bits&=bits-1), O(k) 非 O(n)
# ============================================================
def get_excitation_indices(bits_i, bits_j):
    # 提取 bits_i -> bits_j 的激发: (occupied_list, virtual_list).
    occ = bits_i & ~bits_j  # 在 I 中占据, J 中不占据 -> 被湮灭
    vir = ~bits_i & bits_j  # 在 I 中不占据, J 中占据 -> 被创生
    return bits_to_list(occ), bits_to_list(vir)


# ============================================================
# Rule 1: 对角元 <I|H|I>
# ============================================================
def hii(det_a, det_b, h1e, h2e):
    E = 0.0

    # ---- 单电子: 只遍历置位 ----
    for p in bits_to_list(det_a):
        E += h1e[p, p]
    for p in bits_to_list(det_b):
        E += h1e[p, p]

    # ---- 双电子 ----
    a_occ = bits_to_list(det_a)
    b_occ = bits_to_list(det_b)

    # alpha-alpha: <pq||pq> = h2e[p,p,q,q] - h2e[p,q,q,p]
    for i, p in enumerate(a_occ):
        for q in a_occ[i+1:]:
            E += h2e[p, p, q, q] - h2e[p, q, q, p]

    # beta-beta
    for i, p in enumerate(b_occ):
        for q in b_occ[i+1:]:
            E += h2e[p, p, q, q] - h2e[p, q, q, p]

    # alpha-beta: 无交换项
    for p in a_occ:
        for q in b_occ:
            E += h2e[p, p, q, q]

    return E


# ============================================================
# Rule 2: 单激发 (1 个电子: p -> q)
# ============================================================
def hij_single(a_i, b_i, a_j, b_j, h1e, h2e):
    oa, va = get_excitation_indices(a_i, a_j)
    ob, vb = get_excitation_indices(b_i, b_j)
    if len(oa) + len(ob) != 1:  # 1 个电子激发 (不是 2 个自旋轨道!)
        return 0.0

    if len(oa) == 1:   # alpha 激发: p -> q
        p, q = oa[0], va[0]
        parity = sign(count_between(a_i, p, q))
        same_spin_ref = a_i
        opp_spin_ref = b_i
    else:               # beta 激发: p -> q
        p, q = ob[0], vb[0]
        parity = sign(count_between(b_i, p, q))
        same_spin_ref = b_i
        opp_spin_ref = a_i

    val = h1e[p, q]

    # sum_r <pr||qr> for same-spin -- 用 while+ctz 只遍历置位
    bits = same_spin_ref
    while bits:
        r = ctz(bits)
        bits &= bits - 1
        val += h2e[p, q, r, r] - h2e[p, r, r, q]

    # sum_r <pr|qr> for opposite-spin (无交换)
    bits = opp_spin_ref
    while bits:
        r = ctz(bits)
        bits &= bits - 1
        val += h2e[p, q, r, r]

    return parity * val


# ============================================================
# Rule 3: 双激发 (2 个电子: p->q, r->s)
# ============================================================
def hij_double(a_i, b_i, a_j, b_j, h2e):
    oa, va = get_excitation_indices(a_i, a_j)
    ob, vb = get_excitation_indices(b_i, b_j)
    if len(oa) + len(ob) != 2:  # 2 个电子激发
        return 0.0

    # --- alpha-alpha 双激发: p->q, r->s ---
    if len(oa) == 2 and len(ob) == 0:
        p, r = oa; q, s = va
        # C++ 约定: 第一对宇称用 I, 第二对用 J
        parity = sign(count_between(a_i, p, q)) * sign(count_between(a_j, r, s))
        return parity * (h2e[p, q, r, s] - h2e[p, s, r, q])

    # --- beta-beta 双激发: p->q, r->s ---
    elif len(ob) == 2 and len(oa) == 0:
        p, r = ob; q, s = vb
        parity = sign(count_between(b_i, p, q)) * sign(count_between(b_j, r, s))
        return parity * (h2e[p, q, r, s] - h2e[p, s, r, q])

    # --- alpha-beta 双激发: alpha p->q, beta r->s ---
    elif len(oa) == 1 and len(ob) == 1:
        p, q = oa[0], va[0]
        r, s = ob[0], vb[0]
        parity = sign(count_between(a_i, p, q)) * sign(count_between(b_i, r, s))
        return parity * h2e[p, q, r, s]  # 异自旋无交换

    return 0.0


# ============================================================
# 通用接口: 按 popcount(diff) 分发
# ============================================================
def hij(a_i, b_i, a_j, b_j, h1e, h2e):
    ndiff = popcount(a_i ^ a_j) + popcount(b_i ^ b_j)
    if ndiff == 0:
        return hii(a_i, b_i, h1e, h2e)
    elif ndiff == 2:
        return hij_single(a_i, b_i, a_j, b_j, h1e, h2e)
    elif ndiff == 4:
        return hij_double(a_i, b_i, a_j, b_j, h2e)
    else:
        return 0.0

print("Slater-Condon functions defined.")


Slater-Condon functions defined.


---

## 6 · 枚举行列式 (Gosper's Hack)

按字典序遍历所有"n 选 k"的位串, alpha 和 beta 取笛卡尔积得到所有行列式。


In [20]:
def enumerate_strings(n, k):
    if k == 0:
        yield 0
        return
    x = (1 << k) - 1
    limit = 1 << n
    while x < limit:
        yield x
        # Gosper's hack: next lexicographic combination
        c = x & -x
        r = x + c
        x = (((r ^ x) >> 2) // c) | r

alpha_strings = list(enumerate_strings(norb, nalpha))
beta_strings = list(enumerate_strings(norb, nbeta))

# Cartesian product: all Slater determinants
determinants = [(a, b) for a in alpha_strings for b in beta_strings]
ndet = len(determinants)

print(f"C({norb}, {nalpha}) = {len(alpha_strings)} alpha strings")
print(f"C({norb}, {nbeta})  = {len(beta_strings)} beta strings")
print(f"Total determinants = {ndet}")

# Show a few
print(f"\nFirst 3 alpha strings:")
for a in alpha_strings[:3]:
    print(f"  {a:0{norb}b}")
print(f"HF determinant: alpha={alpha_strings[0]:0{norb}b}, beta={beta_strings[0]:0{norb}b}")


C(7, 5) = 21 alpha strings
C(7, 5)  = 21 beta strings
Total determinants = 441

First 3 alpha strings:
  0011111
  0101111
  0110111
HF determinant: alpha=0011111, beta=0011111


---

## 7 · 构建 Hamiltonian 矩阵


In [21]:
print(f"Building {ndet}x{ndet} Hamiltonian matrix...")

H = np.zeros((ndet, ndet))

for i in range(ndet):
    a_i, b_i = determinants[i]
    H[i, i] = hii(a_i, b_i, h1e, h2e)  # diagonal
    for j in range(i + 1, ndet):
        a_j, b_j = determinants[j]
        val = hij(a_i, b_i, a_j, b_j, h1e, h2e)
        if abs(val) > 1e-14:
            H[i, j] = val
            H[j, i] = val  # Hermitian

nnz = np.count_nonzero(np.abs(H) > 1e-14)
print(f"H shape: {H.shape}")
print(f"Non-zero: {nnz} / {ndet*ndet} ({100*nnz/(ndet*ndet):.1f}%)")

# Verify Hermiticity
herm_err = np.max(np.abs(H - H.T))
print(f"Hermiticity check: max|H - H^T| = {herm_err:.2e}")
assert herm_err < 1e-12, f"H not Hermitian! err={herm_err}"
print("H is Hermitian.")


Building 441x441 Hamiltonian matrix...
H shape: (441, 441)
Non-zero: 18445 / 194481 (9.5%)
Hermiticity check: max|H - H^T| = 0.00e+00
H is Hermitian.


---

## 8 · 对角化 & 与 PySCF FCI 对比


In [22]:
# ============================================================
# 对角化
# ============================================================
eigvals, eigvecs = eigh(H)
E_fci_sc = eigvals[0] + ecore

print(f"Lowest 5 eigenvalues + E_nn (Hartree):")
for i in range(min(5, len(eigvals))):
    print(f"  E[{i}] = {eigvals[i] + ecore:.10f}")

# ============================================================
# PySCF FCI reference
# ============================================================
E_fci_ref, _ = fci_mod.direct_spin1.kernel(
    h1e, h2e, norb, nelec, ecore=ecore, verbose=0
)

print(f"\n{'='*50}")
print(f"Our Slater-Condon FCI:  {E_fci_sc:.10f} Hartree")
print(f"PySCF FCI:              {E_fci_ref:.10f} Hartree")
print(f"Difference:             {abs(E_fci_sc - E_fci_ref):.2e} Hartree")

# RHF comparison
E_rhf = mf.e_tot
E_corr = E_fci_sc - E_rhf
print(f"\nRHF energy:             {E_rhf:.10f} Hartree")
print(f"FCI correlation energy:  {E_corr:.10f} Hartree ({1000*E_corr:.2f} mHartree)")

if abs(E_fci_sc - E_fci_ref) < 1e-8:
    print("\nExact agreement with PySCF FCI!")
else:
    print(f"\nSmall discrepancy: {abs(E_fci_sc - E_fci_ref)*1000:.2f} mHartree")


Lowest 5 eigenvalues + E_nn (Hartree):
  E[0] = -75.0126471190
  E[1] = -74.6147262814
  E[2] = -74.5549978707
  E[3] = -74.5110110018
  E[4] = -74.5090886188

Our Slater-Condon FCI:  -75.0126471190 Hartree
PySCF FCI:              -75.0126471190 Hartree
Difference:             1.22e-12 Hartree

RHF energy:             -74.9630631297 Hartree
FCI correlation energy:  -0.0495839893 Hartree (-49.58 mHartree)

Exact agreement with PySCF FCI!


---

## 9 · 总结

| 层级 | 内容 |
|------|------|
| **积分变换** | `einsum('up,vq,kr,ls,uvkl->pqrs', C, C, C, C, eri_ao)` 不做对称性打包 |
| **行列式** | 整数位运算: `alpha_bits` & `beta_bits` |
| **激发检测** | XOR: `diff = bits_i ^ bits_j`, `popcount(diff)` = 0/2/4 |
| **宇称** | `count_between(bits, p, q)` = p,q 间占据数 |
| **Rule 1** | 对角元 = sum h_{pp} + 1/2 sum <pq\|\|pq> |
| **Rule 2** | 单激发 = Gamma [h_{pq} + sum_r <pr\|\|qr>] |
| **Rule 3** | 双激发 = Gamma*Gamma * <pr\|\|qs> |
| **Rule 4** | >= 3 激发 = 0 (BCH 定理推论) |
| **对角化** | `eigh(H)` dense matrix |
| **验证** | 与 `pyscf.fci.direct_spin1` 对比 |

FCI 是给定基组下的精确解。后续的 MP2、CCSD 将用微扰论/指数 ansatz 来逼近这个能量。
